# Inflection point detection

The inflection points we are looking for are the points where the fluid around
the cylinder changes sign. These will move as vortices are formed and shed. We
are interested in detecting these points and tracking them over time.

Some of the cells in this notebook will have a check if the code is running in
a notebook or is imported. This is because the code is designed to be run in a
notebook, but it can also be imported and run in a script.

## Processing the data

The first step is to post-process the data to extract the inflection points.
This is done by probing the velocity field in a circle around the cylinder and
detecting the points where the velocity changes sign. Each point will be
converted to just an angle and a time, as the radius of the circle is fixed.


In [ ]:
# Assign flags for debugging to alter the behaviour of the program
debug = False
root_folder = "../../../"
log_folder = "logs/cylinder_benchmark/"
data_folder = "results/hpc/cylinder_benchmark/"
experiment = "meshed/"

In [ ]:
# Setup the paths for the project and load the external modules
import os

# Install any missing dependencies and setup the paths for the project
try:
    import numpy as np
    import matplotlib.pyplot as plt

except:
    print("Installing missing dependencies")
    %pip install numpy matplotlib numpy

    import numpy as np
    import matplotlib.pyplot as plt

from metrics.separation_angle import *
from pynektools.io.read_probes import ProbesReader

## Inflection point detection

The inflection points we are looking for are the points where the fluid around
the cylinder changes sign. These will move as vortices are formed and shed. We
are interested in detecting these points and tracking them over time.

### Processing the data

The first step is to post-process the data to extract the inflection points.
This is done by probing the velocity field in a circle around the cylinder and
detecting the points where the velocity changes sign. Each point will be
converted to just an angle and a time, as the radius of the circle is fixed.


In [ ]:
# Load data from the file and plot them to indicate where the probes are located

if debug:
    print("Debug mode enabled")

# Setup the paths for the project
path = os.path.realpath(os.path.join(os.getcwd(), root_folder)) + "/"

if os.path.exists(path + log_folder + experiment):
    path += log_folder + experiment
else:
    path += data_folder + experiment

# Check that the folder exists
if not os.path.exists(path):
    print("The path to the results folder does not exist.")
    print("Please run the benchmark first.")
    print("\tcd NEKO_TOP_ROOT")
    print("\t./setup.sh")
    print("\t./run.sh cylinder_benchmark")
    exit(404)

# Read in the file and setup the data
file_name = "inflection.csv"

# Check that the file exists
if not os.path.exists(file_name):
    file_name = os.path.join(path, file_name)
    if not os.path.exists(file_name):
        raise FileNotFoundError(f"The file: {file_name} does not exist.")

# If the points variable is not initialized, read in the data
if not "is_initialized" in locals() or debug:

    probes = ProbesReader(file_name)

    points = probes.points
    fields = np.asarray([probes.fields["u"], probes.fields["v"]])
    times = probes.times
    field_names = probes.field_names

    N_points = points.shape[0]
    N_fields = fields.shape[1]
    N_times = times.shape[0]
    is_initialized = True
    del probes

# Plot the points in a 2D plot. (Z-axis should be constant)
fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(points[:, 0], points[:, 1])

# Add a circle for the cylinder boundary with radius 0.5
circle = plt.Circle((0, 0), 0.5, color="gray", fill=True)
ax.add_artist(circle)

ax.set_aspect("equal")
plt.title("Probed locations")
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.show()


In [ ]:
# Compute the inflection angles for all time steps
if not "angles" in locals() or debug:
    center = np.array([0.0, 0.0, 0.0])
    angles = track_inflection_point(center, points, fields, times)

fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(times, angles * 360.0 / (2.0 * np.pi))

plt.title("Angle of the inflection points")
plt.xlabel("Time")
plt.ylabel("Angle")
plt.show()

### Detection of stable regions

The next step is to detect the regions where the inflection points are stable.
This is done by examining the time series of the inflection points and
detecting the regions where the oscillations are stable, i.e. the points are
oscillating around a fixed point with a fixed amplitude.

In general we will see 3 regions of interest:

1. The build-up of boundary layers. Here we will see the inflection points move
   from a central point close to the 0 angle to the final stable position.
2. Build-up of vortices. Here we will see the inflection points move from the
   stable position and begin to oscillate with increasing amplitude.
3. Shedding of vortices. Here we will see the inflection points oscillate with
   a large, but stable, amplitude.

#### The layer build-up

The build-up of the boundary layer is the first region of interest. Here we
will see the inflection points move from a central point close to the 0 angle
to the final stable position. We will examine their trajectory and determine
when they have reached a stable position.

#### The vortex build-up

The build-up of vortices is the second region of interest. Here we will see the
inflection points move from the stable position and begin to oscillate with
increasing amplitude. We will examine their amplitude and detect when the
amplitude have reached a stable value.

#### The vortex shedding

The shedding of vortices is the third region of interest. Here we will see the
inflection points oscillate with a large, but stable, amplitude. This is the
main region of interest. This is where we will be examining the frequencies of
the oscillations.

In [ ]:
# Detect the flow stages.
#
# We will be looking for 3 stages in the flow.
# - The boundary layer build up.
# - The vortex formation.
# - The vortex shedding.

if not "i_boundary" in locals() or debug:
    i_boundary, i_building = detect_stable_regions(angles, threshold=1e-6)

print("The layer formation is detected at time: ", times[i_boundary])
print("The vortex building is detected at time: ", times[i_building])

# Extract the data for the stable region
times_stable = times[i_building:]
angles_stable = angles[i_building:, :]

fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(times, angles * 360.0 / (2.0 * np.pi))

ax.axvline(x=times[i_boundary], color="r", linestyle="--")
ax.axvline(x=times[i_building], color="r", linestyle="--")

plt.title("Angle of the inflection points")
plt.xlabel("Time [s]")
plt.ylabel("Angle [deg]")
plt.show()

### Computation of statistics

The final step is to compute the statistics of the stable vortex shedding
region. This will include the frequency of the oscillations, the amplitude of
the oscillations and the phase shift between the different inflection points.

All of these measures are only interesting once the vortex shedding have reached
a stable oscillatory state. This is typically after the boundary layers have
formed and the vortices have started to shed.

1. The frequency is computed using a Fourier transform of the time series of the
   inflection angles.
2. The amplitude is computed as the maximum deviation from the mean of the
   inflection angles.


In [ ]:
# Compute the frequency spectrum of the inflection points
#
# Detect the frequencies present in the inflection point signals display them as
# frequency spectrum. This should allow us to classify the dominant frequencies.

test_times = times[i_building:]
test_angles = angles[i_building:, :]

frequencies, spectrum = compute_vortex_frequency(test_times, test_angles)
max_freq = compute_max_frequency(spectrum, frequencies)
amplitude = compute_amplitudes(test_angles)
bias = compute_bias(test_angles)

print("The dominant frequency for the inflection points are:")
for i in range(len(max_freq)):
    print(f"\t{max_freq[i]:.2f} Hz")

print("The dominant amplitude for the inflection points are:")
for i in range(len(amplitude)):
    print(f"\t{amplitude[i]:.2f} rad")

print("The bias for the inflection points are:")
for i in range(len(bias)):
    print(f"\t{bias[i]:.2f} rad")

fig = plt.figure()
fig.clf()

ax0 = fig.add_subplot(311)
plt.stem(frequencies, np.abs(spectrum[:, 0]), "b", markerfmt=" ", basefmt="-b")
plt.xlim([0, 1])

ax1 = fig.add_subplot(312)
plt.stem(frequencies, np.abs(spectrum[:, 1]), "b", markerfmt=" ", basefmt="-b")
plt.xlim([0, 1])

ax2 = fig.add_subplot(313)
plt.stem(frequencies, np.abs(spectrum[:, 2]), "b", markerfmt=" ", basefmt="-b")
plt.xlim([0, 1])

plt.suptitle("Frequency spectrum of the inflection points")
plt.xlabel("Frequency [Hz]")
plt.show()

## Conclusion and final function

The full benchmarking functipon will be defined bellow. The resulting `dict`
will contain all information relevant to compare multiple flows bast a cylinder.
This concludes this development notebook and one should look into the
`cylinder_benchmark` notebook for the final implementation.


In [ ]:
# Run the inflection benchmark for the meshed cylinder
# if "benchmark_results" not in locals():
benchmark_results = inflection_benchmark(file_name)

print("The dominant frequency for the inflection points are:")
for i in range(len(benchmark_results["max_freq"])):
    print(f"\t{benchmark_results['max_freq'][i]:.2f} Hz")

print("The dominant amplitude for the inflection points are:")
for i in range(len(benchmark_results["amplitude"])):
    print(f"\t{benchmark_results['amplitude'][i]:.2f} rad")

print("The bias for the inflection points are:")
for i in range(len(benchmark_results["bias"])):
    print(f"\t{benchmark_results['bias'][i]:.2f} rad")

In [ ]:
# Now lets try to use the caching mechanism to speed up the computation
import time

# Clear the cache and the benchmark results
benchmark_results = None
os.system("rm -rf cache")

clock_start = time.time()
benchmark_results = inflection_benchmark(file_name)
clock_end = time.time()
print(f"Boundary layer formation: {benchmark_results['i_boundary']}")
t_full = clock_end - clock_start

benchmark_results = None

clock_start = time.time()
benchmark_results = inflection_benchmark(file_name, cache_dir="cache")
clock_end = time.time()
print(f"Boundary layer formation: {benchmark_results['i_boundary']}")
t_cache = clock_end - clock_start

benchmark_results = None

clock_start = time.time()
benchmark_results = inflection_benchmark(file_name, cache_dir="cache")
clock_end = time.time()
print(f"Boundary layer formation: {benchmark_results['i_boundary']}")
t_load = clock_end - clock_start

print(f"Full computation took: {t_full:.2f} seconds")
print(f"Cached computation took: {t_cache:.2f} seconds")
print(f"Loading the cache took: {t_load:.2f} seconds")
